# CardioCare — 01. EDA & 전처리 파이프라인

**데이터셋**: UCI Heart Disease – Cleveland Clinic Foundation  
**버전**: `processed.cleveland.data` (303행 × 13 특성 + 타깃)  
**원본 타깃**: `num` (0 = 정상, 1-4 = 심장병 심각도)  
**이진화 정책**: `target = (num > 0).astype(int)` → 0 = 정상 / 1 = 심장병  
**출처**: https://archive.ics.uci.edu/dataset/45/heart+disease

---
### 노트북 목차
1. [환경 설정 & 데이터 로드](#sec1)
2. [기초 탐색: head / info / describe](#sec2)
3. [타깃 클래스 분포](#sec3)
4. [결측치 분석](#sec4)
5. [이상치 탐지 (IQR + Z-score)](#sec5)
6. [전처리 파이프라인 구성 & 검증](#sec6)
7. [EDA 요약 및 전처리 결정 근거](#sec7)

## 1. 환경 설정 & 데이터 로드<a id='sec1'></a>

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# notebooks/ 에서 실행 시 src/ 를 모듈 경로에 추가
NOTEBOOK_DIR = os.path.abspath('')
ROOT_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 11})

print('Python   :', sys.version.split()[0])
print('pandas   :', pd.__version__)
import sklearn; print('sklearn  :', sklearn.__version__)
print('numpy    :', np.__version__)

In [ ]:
from src.preprocessing import (
    load_raw_data,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    ALL_FEATURES,
    TARGET_COL,
    RANDOM_SEED,
)

DATA_DIR  = os.path.join(ROOT_DIR, 'data')
DATA_PATH = os.path.join(DATA_DIR, 'cleveland.csv')
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    print('[INFO] UCI 아카이브에서 다운로드 중 ...')
    df_raw = load_raw_data()              # URL 에서 읽기
    df_raw.to_csv(DATA_PATH, index=False)
    print(f'[INFO] 저장 완료 -> {DATA_PATH}')
else:
    df_raw = pd.read_csv(DATA_PATH)
    print(f'[INFO] 로컬 파일 로드 -> {DATA_PATH}')

df = df_raw.copy()
print(f'Shape: {df.shape}  (행 x 열)')

## 2. 기초 탐색: head / info / describe <a id='sec2'></a>

In [ ]:
print('=== head(10) ===')
display(df.head(10))

In [ ]:
print('=== info() ===')
df.info()

In [ ]:
print('=== describe() ===')
display(df.describe().round(2))

> **관찰 포인트**  
> - `ca`와 `thal`은 float64 이지만 범주형 값(0~3, 3/6/7)을 갖는다 — 로드 시 `?`가 NaN으로 처리되어 float으로 업캐스트된 것.  
> - `chol`의 최솟값은 126 mg/dl — Cleveland 서브셋에는 0이 없으며, 최대 564 mg/dl의 우측 꼬리 극단값이 이상치 처리 대상이다.  
> - `oldpeak`은 오른쪽으로 치우친 분포(mean > median)이므로 이상치 주의.

## 3. 타깃 클래스 분포 <a id='sec3'></a>

In [ ]:
# 타깃 분포 확인
target_dist = df[TARGET_COL].value_counts(normalize=True).sort_index()
target_cnt  = df[TARGET_COL].value_counts().sort_index()

print('=== 타깃 클래스 분포 ===')
summary = pd.DataFrame({
    'Class':  {0: 'Normal (0)', 1: 'Heart Disease (1)'},
    'Count':  target_cnt,
    'Ratio':  target_dist.map('{:.1%}'.format),
})
display(summary)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = ['Normal (0)', 'Heart Disease (1)']
colors = ['#4C72B0', '#DD8452']

# Bar chart
axes[0].bar(labels, target_cnt.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Class Count')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_cnt.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(
    target_dist.values,
    labels=[f'{l}\n{r:.1%}' for l, r in zip(labels, target_dist.values)],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
)
axes[1].set_title('Class Proportion')

plt.suptitle('Target Class Distribution (Binarized)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'fig1_target_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: data/fig1_target_distribution.png')

> **클래스 분포 → 평가 지표 선택에 미치는 영향**  
>
> Cleveland 데이터셋의 클래스 비율은 **정상 54.5% / 심장병 45.5%** 로 경미한 불균형(mild imbalance)이다.  
> 이 정도의 불균형은 단순 Accuracy 로도 어느 정도 의미 있는 지표를 얻을 수 있지만,  
> **의료 분야의 특성상 False Negative(심장병을 정상으로 잘못 분류)의 임상 비용이 False Positive 보다 훨씬 크다**.  
> 따라서 아래 지표를 우선 채택한다:
> - **Balanced Accuracy**: 클래스 불균형에 강건한 정확도 (두 클래스 Recall 의 산술 평균)
> - **Recall (Sensitivity)**: FN 을 최소화하기 위한 핵심 지표
> - **Precision, F1, Confusion Matrix**: 전체 성능 프로파일 파악
> - **ROC-AUC**: 임계값 선택과 무관한 판별 능력 측정
>
> 불균형이 심화될 경우 SMOTE 오버샘플링이나 `class_weight='balanced'` 옵션 적용을 검토한다.

## 4. 결측치 분석 <a id='sec4'></a>

In [ ]:
# 컬럼별 결측치
missing_per_col = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %':     (df.isnull().mean() * 100).round(2),
    'dtype':         df.dtypes,
}).sort_values('Missing Count', ascending=False)

total_missing = df.isnull().sum().sum()
total_cells   = df.size

print(f'전체 결측 셀: {total_missing} / {total_cells} ({total_missing/total_cells*100:.2f}%)')
print()
display(missing_per_col[missing_per_col['Missing Count'] > 0])

if total_missing == 0:
    print('-> 결측치 없음')

In [ ]:
# 결측치 히트맵
fig, ax = plt.subplots(figsize=(14, 2.5))
sns.heatmap(
    df.isnull().T,
    cbar=False,
    cmap='Reds',
    ax=ax,
    yticklabels=True,
    xticklabels=False,
)
ax.set_title('Missing Value Heatmap (Red = Missing)', fontsize=12)
ax.set_xlabel('Samples')
plt.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'fig2_missing_heatmap.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: data/fig2_missing_heatmap.png')

> **결측치 처리 결정**  
>
> | 컬럼 | 결측 수 | 결측률 | 처리 방법 | 근거 |
> |------|---------|--------|-----------|------|
> | `ca` | 4 | 1.3% | 중앙값 대치 | 결측 비율이 매우 낮고, 값이 순서형(0-3)이라 중앙값이 적합 |
> | `thal` | 2 | 0.7% | 최빈값 대치 | 범주형(3/6/7)이므로 최빈값이 적합 |
>
> **행 삭제를 선택하지 않은 이유**: 결측 행이 6개(2%)뿐이며, Cleveland 데이터셋은 303행으로 소규모여서 행 삭제 시 학습 데이터 손실이 상대적으로 크다.  
> KNN Imputer도 고려했으나, 결측 비율이 충분히 낮아 단순 대치로도 성능 차이가 미미하고 재현성·속도 면에서 유리한 median/mode 대치를 채택한다.

## 5. 이상치 탐지 — 연속형 특성 <a id='sec5'></a>

대상 특성: `age`, `trestbps`, `chol`, `thalach`, `oldpeak`

In [ ]:
CONT_FEATS = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

# --- Boxplot ---
fig, axes = plt.subplots(1, len(CONT_FEATS), figsize=(16, 5))
for ax, feat in zip(axes, CONT_FEATS):
    series = df[feat].dropna()
    bp = ax.boxplot(
        series,
        vert=True,
        patch_artist=True,
        notch=False,
        boxprops=dict(facecolor='#AEC6CF', color='#2c3e50'),
        medianprops=dict(color='#e74c3c', linewidth=2),
        flierprops=dict(marker='o', markerfacecolor='#e74c3c', markersize=5, alpha=0.6),
        whiskerprops=dict(color='#2c3e50'),
        capprops=dict(color='#2c3e50'),
    )
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xticks([])

plt.suptitle('Boxplots — Continuous Features (Outlier Detection)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'fig3_boxplots.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: data/fig3_boxplots.png')

In [ ]:
# --- IQR 기반 이상치 개수 및 경계 ---
print(f'{'Feature':<12} {'N_outlier':>9} {'Lower':>9} {'Upper':>9} {'IQR':>8}')
print('-' * 52)
iqr_results = {}
for feat in CONT_FEATS:
    series = df[feat].dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR_val = Q3 - Q1
    lower = Q1 - 1.5 * IQR_val
    upper = Q3 + 1.5 * IQR_val
    n_out = int(((series < lower) | (series > upper)).sum())
    pct   = n_out / len(series) * 100
    iqr_results[feat] = {'lower': lower, 'upper': upper, 'n_outlier': n_out}
    print(f'{feat:<12} {n_out:>5} ({pct:4.1f}%) {lower:>9.1f} {upper:>9.1f} {IQR_val:>8.1f}')

In [ ]:
# --- Z-score 기반 이상치 개수 (|z| > 3) ---
print('Z-score 기반 이상치 개수 (|z| > 3):')
print(f'  {'Feature':<12} {'N_outlier':>9} {'Max |z|':>9}')
print('  ' + '-' * 34)
for feat in CONT_FEATS:
    series = df[feat].dropna()
    z = np.abs(stats.zscore(series))
    n_out = int((z > 3).sum())
    print(f'  {feat:<12} {n_out:>9d} {z.max():>9.2f}')

In [ ]:
# --- 분포 히스토그램 + KDE ---
fig, axes = plt.subplots(2, len(CONT_FEATS), figsize=(18, 8))
for i, feat in enumerate(CONT_FEATS):
    series = df[feat].dropna()
    # 정상(0) vs 심장병(1) 분리
    s0 = df[df[TARGET_COL] == 0][feat].dropna()
    s1 = df[df[TARGET_COL] == 1][feat].dropna()

    # 상단: 전체 히스토그램 + IQR 경계
    ax_top = axes[0, i]
    ax_top.hist(series, bins=25, color='steelblue', alpha=0.7, edgecolor='white')
    lo, hi = iqr_results[feat]['lower'], iqr_results[feat]['upper']
    ax_top.axvline(lo, color='red',    linestyle='--', linewidth=1.2, label=f'IQR lb={lo:.0f}')
    ax_top.axvline(hi, color='orange', linestyle='--', linewidth=1.2, label=f'IQR ub={hi:.0f}')
    ax_top.set_title(feat, fontsize=10)
    ax_top.legend(fontsize=7)

    # 하단: 클래스별 KDE
    ax_bot = axes[1, i]
    s0.plot.kde(ax=ax_bot, label='Normal(0)',       color='#4C72B0', linewidth=1.8)
    s1.plot.kde(ax=ax_bot, label='HeartDisease(1)', color='#DD8452', linewidth=1.8)
    ax_bot.legend(fontsize=7)
    ax_bot.set_xlabel(feat, fontsize=9)

axes[0, 0].set_ylabel('Count')
axes[1, 0].set_ylabel('Density')
plt.suptitle('Feature Distributions — Histogram (top) & Class KDE (bottom)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'fig4_distributions.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: data/fig4_distributions.png')

> **이상치 관찰 요약**  
>
> | 특성 | 주요 이상치 패턴 | 처리 방법 |
> |------|----------------|----------|
> | `chol` | 우측 꼬리 고값 (최대 564 mg/dl) | IQR 클리핑 (k=1.5) |
> | `trestbps` | 우측 꼬리 — 200 mmHg 초과 몇 개 | IQR 클리핑 |
> | `thalach` | 좌측 꼬리 — 기저 심박수가 매우 낮은 케이스 | IQR 클리핑 |
> | `oldpeak` | 강한 오른쪽 치우침(skew) — 최댓값 6.2 | IQR 클리핑 |
> | `age` | 분포 비교적 정상, 이상치 소수 | IQR 클리핑 |
>
> **IQR Clipping 선택 근거**: 행 삭제 시 소규모 데이터셋에서 학습 손실이 크다.  
> Z-score 방식은 정규성을 가정하므로 `oldpeak` 같은 치우친 분포에 부적합.  
> IQR k=1.5 은 Tukey fence 표준으로, `IQROutlierClipper`가 **학습 데이터의 Q1/Q3만 사용**하므로 누수가 없다.

## 6. 전처리 파이프라인 구성 & 검증 <a id='sec6'></a>

`src/preprocessing.py` 의 `build_preprocessing_pipeline()` 을 사용한다.  
**핵심 원칙**: `train_test_split` 을 먼저 수행하고, 그 이후에만 파이프라인을 `.fit()` 한다.

In [ ]:
from sklearn.model_selection import train_test_split
from src.preprocessing import build_preprocessing_pipeline

X = df[ALL_FEATURES].copy()
y = df[TARGET_COL].copy()

# ── Step 1: split BEFORE any fitting (누수 방지) ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,          # 80 / 20 분할
    random_state=RANDOM_SEED,
    stratify=y,              # 클래스 비율 유지
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train target ratio: {y_train.mean():.3f}  |  Test target ratio: {y_test.mean():.3f}')

In [ ]:
# ── Step 2: 파이프라인 구성 ────────────────────────────────────────────────
preprocessor = build_preprocessing_pipeline(impute_strategy='median')
print('파이프라인 구성:')
print(preprocessor)

# ── Step 3: 학습 데이터에만 fit → test 는 transform only ─────────────────
X_train_proc = preprocessor.fit_transform(X_train)   # fit + transform
X_test_proc  = preprocessor.transform(X_test)        # transform ONLY — 누수 없음

print(f'\nX_train 전처리 전: {X_train.shape}')
print(f'X_train 전처리 후: {X_train_proc.shape}')
print(f'X_test  전처리 후: {X_test_proc.shape}')

In [ ]:
# ── 검증 1: 변환 후 NaN 잔존 여부 ────────────────────────────────────────
assert not np.isnan(X_train_proc).any(), 'Train에 NaN 잔존!'
assert not np.isnan(X_test_proc).any(),  'Test에 NaN 잔존!'
print('[PASS] 전처리 후 NaN 없음')

# ── 검증 2: 연속형 특성이 표준화되었는지 확인 ─────────────────────────────
n_num = len(NUMERIC_FEATURES)   # 6개
train_mean = X_train_proc[:, :n_num].mean(axis=0)
train_std  = X_train_proc[:, :n_num].std(axis=0)
print(f'\n[연속형 특성 통계 — 학습셋]')
for i, feat in enumerate(NUMERIC_FEATURES):
    print(f'  {feat:<10}: mean={train_mean[i]:+.4f}  std={train_std[i]:.4f}')
print('(표준화 후 mean≈0, std≈1 기대)')

# ── 검증 3: 테스트셋 통계는 학습셋과 약간 다름 (정상적) ────────────────────
test_mean = X_test_proc[:, :n_num].mean(axis=0)
print(f'\n[연속형 특성 mean — 테스트셋 (학습 scaler 적용)]')
for i, feat in enumerate(NUMERIC_FEATURES):
    print(f'  {feat:<10}: mean={test_mean[i]:+.4f}  (0에 가까울수록 분포 유사)')

In [ ]:
# ── 파이프라인 결정론성 검증 (동일 입력 → 동일 출력) ──────────────────────
preprocessor2 = build_preprocessing_pipeline(impute_strategy='median')
X_train_proc2 = preprocessor2.fit_transform(X_train)
assert np.allclose(X_train_proc, X_train_proc2), '파이프라인이 비결정론적!'
print('[PASS] 파이프라인 결정론적 확인 (동일 입력 → 동일 출력)')

In [ ]:
# ── 전처리 전후 분포 비교 (연속형 3개) ────────────────────────────────────
sample_feats = ['chol', 'trestbps', 'oldpeak']
train_df = pd.DataFrame(X_train[sample_feats].values, columns=sample_feats)
col_indices = [NUMERIC_FEATURES.index(f) for f in sample_feats]
train_proc_df = pd.DataFrame(X_train_proc[:, col_indices], columns=sample_feats)

fig, axes = plt.subplots(2, len(sample_feats), figsize=(14, 7))
for i, feat in enumerate(sample_feats):
    axes[0, i].hist(train_df[feat].dropna(), bins=25, color='steelblue',
                    alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'{feat} (before)', fontsize=10)
    axes[1, i].hist(train_proc_df[feat], bins=25, color='#55A868',
                    alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'{feat} (after: clipped + scaled)', fontsize=10)

axes[0, 0].set_ylabel('Count (raw)')
axes[1, 0].set_ylabel('Count (scaled)')
plt.suptitle('Distribution Before vs After Preprocessing (Train set)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(DATA_DIR, 'fig5_before_after_preprocess.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: data/fig5_before_after_preprocess.png')

## 7. EDA 요약 및 전처리 결정 근거 <a id='sec7'></a>

---

### EDA 가 무엇을 알려 주었으며, 그에 따라 전처리에서 무엇을 어떻게 바꾸었는가?

#### 1. 클래스 분포
- 정상 54.5% / 심장병 45.5% 로 **경미한 불균형**이다.  
  단순 Accuracy 를 주지표로 삼으면 다수 클래스에 편향될 수 있으므로,  
  **Balanced Accuracy, Recall, F1** 을 우선 지표로 채택했다.

#### 2. 결측치
- 전체 결측률 < 2% 이며 `ca`(4개), `thal`(2개) 두 컬럼에만 집중.  
  행 삭제 시 학습 손실이 상대적으로 크기 때문에 **대치(Imputation)** 를 선택했다.  
  - `ca` (순서형 0-3): 중앙값 대치 — 이상치에 강건하고 계산이 단순.  
  - `thal` (범주형 3/6/7): 최빈값 대치 — 범주형 변수에 적합.

#### 3. 이상치
- `chol`: 최대 564 mg/dl의 극단값이 있고 우측 꼬리 분포 → IQR 클리핑 필수.  
- `oldpeak`: 강한 오른쪽 치우침(skewness) → Z-score 방식보다 IQR 방식이 적합.  
- `trestbps`, `thalach`: 임상적으로 납득 가능한 극단값이 소수 존재 → IQR 클리핑으로 완화.  
  **대응**: `IQROutlierClipper(k=1.5)` 를 파이프라인 내 imputer 뒤, scaler 앞에 배치.

#### 4. 특성 스케일링
- 연속형 특성은 값의 범위가 크게 다르므로(age: 29-77, chol: 126-564)  
  **StandardScaler** 를 적용하여 거리·그래디언트 기반 모델(SVC, Logistic Regression)의 성능을 보호.

#### 5. 범주형 인코딩
- `cp`, `restecg`, `slope`, `thal` 은 명목/순서 범주형 → **OneHotEncoder** 로 처리.  
  명목 변수에 OrdinalEncoder 를 쓰면 모델이 존재하지 않는 순서 관계를 학습할 위험이 있다.

#### 6. 누수 방지 설계
- Imputer, Clipper, Scaler, Encoder 모두 **`build_preprocessing_pipeline()`** 내에 캡슐화.  
  파이프라인은 `train_test_split` 이후 `X_train` 에만 `.fit()` 하므로  
  테스트셋 정보가 어떤 형태로도 학습 과정에 유입되지 않는다.

---
*다음 단계*: `notebooks/02_train.py` 에서 이 파이프라인을 모델 학습 파이프라인에 통합하고 MLflow 로 실험을 추적한다.